# Notebook 36 — Multi-Agent Workflows and Coordination

    ## Learning objectives

    - Implement supervisor-worker, handoff, parallel fan-out, debate, and specialist patterns
- Define typed delegation, state isolation, permissions, budgets, and deterministic aggregation
- Evaluate whether multiple agents improve outcomes enough to justify coordination cost

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if IN_COLAB and not token:
    from google.colab import userdata
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            token = None
        if token:
            break
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub variable. The course also sets its
# descriptive alias because some lesson code uses HUGGINGFACE_TOKEN explicitly.
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGINGFACE_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("Add an HF_TOKEN secret in Colab, enable notebook access, then rerun this cell.")


## 36.1 What “multi-agent” means in a workflow

A multi-agent system gives two or more model-controlled roles distinct context, instructions, tools, or authority
and coordinates their outputs. It may be a supervisor delegating subtasks, a sequence of specialist handoffs,
parallel workers with an aggregator, or independent proposal and critique. Multiple labels in one prompt are not
operational isolation; real boundaries require separate state and capability enforcement.

Start with a deterministic workflow or one agent with multiple tools. Add agents when specialization, context
isolation, parallelizable work, independent verification, or different permission domains produce measured value.
Additional agents multiply tokens, latency, correlated errors, prompt-injection paths, and debugging complexity.


In [ ]:
patterns = {
    "supervisor-worker":"dynamic delegation and synthesis",
    "sequential handoff":"specialist owns the next typed state transition",
    "parallel fan-out":"independent subtasks followed by deterministic aggregation",
    "proposal-critic":"separate candidate generation and verification",
    "committee/debate":"diverse judgments with an explicit stopping/selection rule",
}
for name, purpose in patterns.items(): print(f"{name:20} | {purpose}")


## 36.2 Delegation is a typed contract

A task envelope includes objective, inputs by reference, constraints, allowed tools, output schema, budget,
deadline, provenance, parent run, and success criteria. Workers receive least privilege and only necessary context.
Their result includes status, structured output, evidence, uncertainty, resource use, and errors. Natural-language
delegation alone invites scope drift and makes retries ambiguous.

The supervisor may select workers but cannot grant capabilities it does not possess. Identity and tenant context
come from the host. Delegation depth, fan-out, total calls, tokens, wall time, and spend need global and per-worker
limits. A worker cannot recursively create agents unless explicitly allowed.


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class TaskEnvelope:
    task_id: str; role: str; objective: str; allowed_tools: tuple[str,...]
    max_steps: int; max_tokens: int; parent_run: str
tasks = [TaskEnvelope("t1","retriever","find supporting sources",("search",),3,1200,"run-1"),
         TaskEnvelope("t2","analyst","check numerical claims",("calculator",),3,1200,"run-1")]
print(tasks)


## 36.3 Parallel specialists and deterministic aggregation

Fan out only independent work. Workers should not mutate shared state concurrently; collect immutable results and
merge in a deterministic node. Preserve ordering and worker identity. Deduplicate evidence, resolve schema-level
conflicts, and escalate substantive disagreement rather than asking an aggregator to conceal it in fluent prose.

Parallelism reduces wall time only when model/tool infrastructure supports concurrency and shared bottlenecks do
not dominate. Bound simultaneous requests and cancel siblings when the task is resolved. Partial failure policy—
fail closed, return partial evidence, retry, or use a fallback—belongs in the workflow contract.


In [ ]:
import asyncio
async def worker(task):
    await asyncio.sleep(0.01)
    return {"task_id":task.task_id, "role":task.role, "status":"ok", "claims":[task.objective]}
results = await asyncio.gather(*(worker(task) for task in tasks))
merged = {"status":"ok" if all(r["status"]=="ok" for r in results) else "partial",
          "results":sorted(results, key=lambda r:r["task_id"])}
print(merged)


## 36.4 Supervisors, handoffs, and shared state

Supervisors decompose work and route results, but an unconstrained supervisor can loop, duplicate tasks, or route
sensitive data incorrectly. Restrict available roles and validate every envelope. Sequential handoffs are useful
when research, drafting, review, and approval own different state transitions; the next role receives a compact
typed artifact rather than the entire hidden transcript.

Shared blackboards simplify coordination but become injection and race-condition surfaces. Prefer append-only
events and namespaced worker scratch space. Separate authoritative workflow state from agent observations. Use
optimistic concurrency for updates and make ownership of each field explicit.


In [ ]:
routing = {"needs_sources":"retriever", "needs_math":"analyst", "ready_to_write":"writer",
           "consequential_action":"human_approval"}
proposed_state = "consequential_action"
next_owner = routing[proposed_state]
print("deterministic boundary routes to:", next_owner)


## 36.5 Critics, debate, and diversity

A critic should use an independent rubric or verifier, not merely echo the proposer. Blind worker identity and
randomize ordering to reduce position and authority bias. Debate can surface assumptions but can also converge on
persuasive shared errors. Diversity requires different evidence, prompts, models, tools, or seeds—not different
job titles alone.

Define a stopping rule and selector before the discussion. Prefer deterministic tests for code, arithmetic,
schemas, and citations. Preserve unresolved dissent in the final result. Never use majority vote when all agents
share the same flawed source or when one safety failure is disqualifying.


In [ ]:
proposals = [{"worker":"a","answer":42,"evidence":{"test-1"}},
             {"worker":"b","answer":42,"evidence":{"test-2"}},
             {"worker":"c","answer":41,"evidence":set()}]
verified = [p for p in proposals if p["answer"] == 42 and p["evidence"]]
print("accepted:", verified, "unresolved dissent:", [p for p in proposals if p not in verified])


## 36.6 Multi-agent security and evaluation

Give workers distinct service identities, tool allowlists, data scopes, and network boundaries. Do not forward one
worker's credentials or raw context to another. Label messages by authenticated origin outside model text. Treat
every inter-agent message as untrusted; prevent a compromised retriever from instructing a deployment worker.
Consequential actions pass through deterministic authorization and human approval regardless of consensus.

Compare multi-agent against single-agent and deterministic baselines under matched models, token/cost budgets, and
time. Measure task success, critical failures, tool accuracy, duplicated work, conflicts, fan-out, depth, total
tokens, latency, intervention, and recovery. Ablate each role. If removing an agent does not hurt frozen evaluation,
remove the complexity. This notebook therefore covers the multi-agent workflows commonly used in practice while
treating coordination as an evidence-driven systems decision.


## 36.7 When multiple agents are justified

Add agents only when separate context, permissions, failure domains, parallel latency, or independently evaluated expertise provides measurable value. Role-playing several agents on one shared transcript often adds cost and correlated error without real isolation. Compare deterministic workflow, single agent with tools, and multi-agent architecture under equal model, token, tool, time, and permission budgets. Retain the simplest design that passes the same gates.


In [ ]:
architectures=[{"name":"single","success":.80,"tokens":1000,"violations":0},{"name":"multi","success":.83,"tokens":2400,"violations":0}];
for a in architectures: print(a["name"],"success per 1k tokens",a["success"]/(a["tokens"]/1000))


## 36.8 Typed task envelopes and deterministic merge

Delegation messages should contain task ID, objective, inputs or references, output schema, deadline, budgets, permissions, and parent trace. Workers cannot broaden scope. Parallel results need a deterministic merge rule that preserves provenance, conflicts, uncertainty, and dissent; asking another model to blend them can erase evidence. Propagate cancellation and global budgets across the tree. Detect cycles, duplicated work, stale results, and specialists attempting unauthorized tools.


In [ ]:
results=[{"agent":"a","claim":"x","confidence":.8,"source":"d1"},{"agent":"b","claim":"not x","confidence":.7,"source":"d2"}]; grouped={}
for r in results: grouped.setdefault(r["claim"],[]).append(r)
print("preserved alternatives",grouped)


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [AutoGen](https://arxiv.org/abs/2308.08155)
- [CAMEL](https://arxiv.org/abs/2303.17760)


## Exercises

    1. Build a supervisor with a strict role registry and reject recursive or over-budget delegation.
2. Compare parallel specialists with one agent under equal total token and tool-call budgets.
3. Inject a malicious worker message and prove it cannot expand another worker's permissions.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
